In [ ]:
pip install opencv-python pytesseract pandas


In [ ]:

!apt update
!apt install -y tesseract-ocr
!apt install -y libtesseract-dev

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,631 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,226 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packa

In [ ]:

import cv2
import numpy as np
import pytesseract
import pandas as pd
import re
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

In [ ]:

import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'
!tesseract --version

tesseract 4.1.1
 leptonica-1.82.0
  libgif 5.1.9 : libjpeg 8d (libjpeg-turbo 2.1.1) : libpng 1.6.37 : libtiff 4.3.0 : zlib 1.2.11 : libwebp 1.2.2 : libopenjp2 2.4.0
 Found AVX2
 Found AVX
 Found FMA
 Found SSE
 Found libarchive 3.6.0 zlib/1.2.11 liblzma/5.2.5 bz2lib/1.0.8 liblz4/1.9.3 libzstd/1.4.8


In [ ]:

# Step 1: Preprocess the Image
def preprocess_image(image_path):
    image = cv2.imread(image_path)

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Enhance contrast using CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Apply adaptive thresholding
    binary = cv2.adaptiveThreshold(
        enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 2
    )

    # Morphological transformations to strengthen table lines
    kernel = np.ones((2, 2), np.uint8)
    morph = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    return morph, image


In [ ]:

# Step 2: Detect Table and Extract Cells
def detect_table_structure(binary_image):
    # Extract horizontal and vertical lines
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 1))
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 30))

    horizontal_lines = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, horizontal_kernel)
    vertical_lines = cv2.morphologyEx(binary_image, cv2.MORPH_OPEN, vertical_kernel)

    # Combine lines to detect the table structure
    table_structure = cv2.add(horizontal_lines, vertical_lines)
    return table_structure

def extract_cells_and_text(image, table_structure):
    contours, _ = cv2.findContours(
        table_structure, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )

    # Sort contours by their position (top-to-bottom, then left-to-right)
    sorted_contours = sorted(
        contours, key=lambda c: (cv2.boundingRect(c)[1], cv2.boundingRect(c)[0])
    )

    table_data = []
    current_row = []
    previous_y = -1

    for contour in sorted_contours:
        x, y, w, h = cv2.boundingRect(contour)

        # Ignore small boxes (noise)
        if w < 30 or h < 20:
            continue

        # Extract the cell image
        cell = image[y:y+h, x:x+w]
        text = pytesseract.image_to_string(cell, config="--psm 6").strip()

        # Detect new row based on y-coordinate
        if previous_y != -1 and abs(y - previous_y) > 10:
            table_data.append(current_row)
            current_row = []

        # Append cleaned text to the current row
        current_row.append(clean_text(text))
        previous_y = y

    # Append the last row
    if current_row:
        table_data.append(current_row)

    return table_data


In [ ]:

# Step 3: Clean OCR Text
def clean_text(text):
    # Remove unwanted characters and normalize text
    text = re.sub(r"[^a-zA-Z0-9\s.,_-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:

# Step 4: Filter and Save to CSV
def normalize_and_save(table_data, output_path):
    # Find the maximum number of columns
    max_columns = max(len(row) for row in table_data)

    # Normalize rows (pad with empty strings)
    normalized_data = [row + [""] * (max_columns - len(row)) for row in table_data]

    # Create a DataFrame
    df = pd.DataFrame(normalized_data)

    # Remove the specific noisy row based on its pattern
    df = df[~df.apply(lambda row: row.astype(str).str.cat(sep=" ").startswith("A B c Do"), axis=1)]

    # Save to CSV
    df.to_csv(output_path, index=False)
    print(f"CSV saved to {output_path}")

# GUI Widgets for Upload and Download
upload_button = widgets.FileUpload(accept='.png,.jpg,.jpeg', multiple=False, button_style="info")
output_area = widgets.Output()
download_button = widgets.Button(description="Download CSV", button_style="success")

# Style buttons and layout
upload_button.style.button_color = "blue"
download_button.style.button_color = "green"

# Container for layout
container = widgets.VBox([
    widgets.HTML("<h1 style='color: red; text-align: center;'>Image to CSV Converter</h1>"),  # Use widgets.HTML
    upload_button,
    output_area,
    download_button
], layout=widgets.Layout(border='solid 2px black', padding='10px', align_items='center'))


In [ ]:

# GUI Widgets for Upload and Download
upload_button = widgets.FileUpload(accept='.png,.jpg,.jpeg', multiple=False, button_style="info")
output_area = widgets.Output()
download_button = widgets.Button(description="Download CSV", button_style="success")

# Function to handle file upload
def handle_upload(change):
    with output_area:
        clear_output()
        uploaded_file = upload_button.value
        if not uploaded_file:
            print("No file uploaded. Please try again.")
            return

        file_info = list(uploaded_file.values())[0]
        file_path = "/content/" + file_info["metadata"]["name"]
        with open(file_path, 'wb') as f:
            f.write(file_info["content"])

        try:
            # Preprocess the image
            binary_image, original_image = preprocess_image(file_path)

            # Detect the table structure
            table_structure = detect_table_structure(binary_image)

            # Extract table data
            table_data = extract_cells_and_text(original_image, table_structure)

            # Save to CSV
            output_csv_path = "/content/extracted_table.csv"
            normalize_and_save(table_data, output_csv_path)

            # Clear existing event handlers for the download button
            download_button.on_click(lambda _: None)  # Detach all handlers

            # Attach a new click handler for downloading the file
            def download_file(b):
                files.download(output_csv_path)

            download_button.on_click(download_file)  # Attach handler
            print("CSV ready for download. Click the button below:")
            display(download_button)

        except Exception as e:
            print(f"Error: {e}")

# Attach file upload handler
upload_button.observe(handle_upload, names="value")

# Layout: Adding title, buttons, and styles
container = widgets.VBox([
    widgets.HTML("<h1 style='color: red; text-align: center;'>Image to CSV Converter</h1>"),
    upload_button,
    output_area,
], layout=widgets.Layout(border='solid 2px black', padding='10px', align_items='center'))

# Display the interface
display(container)


### **PDF to CSV**

In [ ]:
!pip install pdf2image
!apt-get install -y poppler-utils
!pip install pdfplumber


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.5).
0 upgraded, 0 newly installed, 0 to remove and 50 not upgraded.


In [ ]:
import pdfplumber
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

In [ ]:

# Function to extract table from PDF
def extract_table_from_pdf(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        all_data = []
        for page in pdf.pages:
            # Extract tables from each page
            tables = page.extract_tables()
            for table in tables:
                # Combine all table data
                all_data.extend(table)
        return all_data

# Function to normalize and save table data to CSV
def normalize_and_save_to_csv(table_data, output_path):
    # Remove empty rows and normalize the table
    normalized_data = [row for row in table_data if any(row)]
    max_columns = max(len(row) for row in normalized_data)
    normalized_data = [row + [""] * (max_columns - len(row)) for row in normalized_data]

    # Create a DataFrame
    df = pd.DataFrame(normalized_data)

    # Save to CSV
    df.to_csv(output_path, index=False, header=False)
    print(f"CSV saved to {output_path}")


In [ ]:

# GUI Components
upload_button = widgets.FileUpload(accept='.pdf', multiple=False)
output_area = widgets.Output()
download_button = widgets.Button(description="Download CSV", button_style="success")
output_path = "extracted_table.csv"

# Button Handlers
def handle_file_upload(change):
    with output_area:
        clear_output()
        try:
            uploaded_file = list(upload_button.value.values())[0]
            content = uploaded_file['content']
            file_path = '/content/temp.pdf'
            with open(file_path, 'wb') as f:
                f.write(content)

            # Extract table data from PDF
            table_data = extract_table_from_pdf(file_path)

            # Normalize and save to CSV
            normalize_and_save_to_csv(table_data, output_path)

            # Provide download link
            print("CSV is ready for download.")
        except Exception as e:
            print(f"Error: {e}")


In [ ]:
def download_csv(change):
    files.download(output_path)

# Event Listeners
upload_button.observe(handle_file_upload, names='value')
download_button.on_click(download_csv)

# Layout
container = widgets.VBox([
    widgets.HTML("<h1 style='color: red; text-align: center;'>PDF to CSV Converter</h1>"),
    upload_button,
    output_area,
    download_button
], layout=widgets.Layout(border='solid 2px black', padding='10px', align_items='center'))

display(container)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>